# Learning Objectives
In this notebook, you will learn Spark Dataframe APIs.

# Question List

Solve the following questions using Spark Dataframe APIs

### Join

1. easy - https://pgexercises.com/questions/joins/simplejoin.html
2. easy - https://pgexercises.com/questions/joins/simplejoin2.html
3. easy - https://pgexercises.com/questions/joins/self2.html 
4. medium - https://pgexercises.com/questions/joins/threejoin.html (three join)
5. medium - https://pgexercises.com/questions/joins/sub.html (subquery and join)

### Aggregation

1. easy - https://pgexercises.com/questions/aggregates/count3.html Group by order by
2. easy - https://pgexercises.com/questions/aggregates/fachours.html group by order by
3. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth.html group by with condition 
4. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth2.html group by multi col
5. easy - https://pgexercises.com/questions/aggregates/members1.html count distinct
6. med - https://pgexercises.com/questions/aggregates/nbooking.html group by multiple cols, join

### String & Date

1. easy - https://pgexercises.com/questions/string/concat.html format string
2. easy - https://pgexercises.com/questions/string/case.html WHERE + string function
3. easy - https://pgexercises.com/questions/string/reg.html WHERE + string function
4. easy - https://pgexercises.com/questions/string/substr.html group by, substr
5. easy - https://pgexercises.com/questions/date/series.html generate ts
6. easy - https://pgexercises.com/questions/date/bookingspermonth.html extract month from ts


# Setup

Run this cell first. It imports the Spark functions we'll use throughout, and loads
the three managed tables created in `0 - ETL pgexercieses CSV files` into DataFrames.

- `bookings`: bookid, facid, memid, starttime, slots
- `facilities`: facid, name, membercost, guestcost, initialoutlay, monthlymaintenance
- `members`: memid, surname, firstname, address, zipcode, telephone, recommendedby, joindate


In [ ]:
from pyspark.sql.functions import (
    col, concat_ws, count, countDistinct, sum as _sum, min as _min,
    when, lower, substring, month, year, date_trunc, sequence, explode,
    to_date, lit, expr
)

bookings = spark.table("bookings")
facilities = spark.table("facilities")
members = spark.table("members")


## Join 1

### Question

How can you produce a list of the start times for bookings by members named 'David Farrell'?

https://pgexercises.com/questions/joins/simplejoin.html


In [ ]:
df = (
    bookings.alias("b")
    .join(members.alias("m"), col("b.memid") == col("m.memid"))
    .filter((col("m.firstname") == "David") & (col("m.surname") == "Farrell"))
    .select(col("b.starttime"))
)
display(df)


## Join 2

### Question

How can you produce a list of the start times for bookings for tennis courts, for the date
'2012-09-21'? Return a list of start time and facility name pairings, ordered by the time.

https://pgexercises.com/questions/joins/simplejoin2.html


In [ ]:
df = (
    bookings.alias("b")
    .join(facilities.alias("f"), col("b.facid") == col("f.facid"))
    .filter(
        col("f.name").like("Tennis Court%")
        & (col("b.starttime") >= "2012-09-21")
        & (col("b.starttime") < "2012-09-22")
    )
    .select(col("b.starttime").alias("start"), col("f.name").alias("name"))
    .orderBy("start")
)
display(df)


## Join 3

### Question

How can you output a list of all members, including the individual who recommended them
(if any)? Ensure that results are ordered by (surname, firstname).

https://pgexercises.com/questions/joins/self2.html


In [ ]:
recommenders = members.alias("recs")

df = (
    members.alias("m")
    .join(recommenders, col("m.recommendedby") == col("recs.memid"), "left")
    .select(
        col("m.firstname").alias("memfname"),
        col("m.surname").alias("memsname"),
        col("recs.firstname").alias("recfname"),
        col("recs.surname").alias("recsname"),
    )
    .orderBy("memsname", "memfname")
)
display(df)


## Join 4 (three join)

### Question

How can you produce a list of all members who have used a tennis court? Include in your
output the name of the court, and the name of the member formatted as a single column.
Ensure no duplicate data, and order by the member name.

https://pgexercises.com/questions/joins/threejoin.html


In [ ]:
df = (
    bookings
    .join(members, "memid")
    .join(facilities, "facid")
    .filter(col("name").like("Tennis Court%"))
    .select(
        concat_ws(" ", col("firstname"), col("surname")).alias("member"),
        col("name").alias("facility"),
    )
    .distinct()
    .orderBy("member")
)
display(df)


## Join 5 (subquery and join)

### Question

How can you output a list of all members, including the individual who recommended them
(if any), without using any joins? Ensure that there are no duplicates in the list, and that
each firstname + surname pairing is formatted as a column and ordered.

https://pgexercises.com/questions/joins/sub.html

Note: the DataFrame API doesn't have a true "no-join" equivalent to a correlated subquery -
a `join` (here a `left` join to emulate the optional recommender) is the natural DataFrame
way to express this. We use `when`/`otherwise` to null out the recommender when there isn't one.


In [ ]:
recommenders = members.alias("recs")

df = (
    members.alias("m")
    .join(recommenders, col("m.recommendedby") == col("recs.memid"), "left")
    .select(
        concat_ws(" ", col("m.firstname"), col("m.surname")).alias("member"),
        when(col("recs.memid").isNull(), None)
        .otherwise(concat_ws(" ", col("recs.firstname"), col("recs.surname")))
        .alias("recommender"),
    )
    .distinct()
    .orderBy("member")
)
display(df)


## Aggregation 1

### Question

Produce a count of the number of recommendations each member has made. Order by member ID.

https://pgexercises.com/questions/aggregates/count3.html


In [ ]:
df = (
    members
    .filter(col("recommendedby").isNotNull())
    .groupBy("recommendedby")
    .agg(count("*").alias("count"))
    .orderBy("recommendedby")
)
display(df)


## Aggregation 2

### Question

Produce a list of the total number of slots booked per facility. For now, just produce an
output table consisting of facility id and slots, sorted by facility id.

https://pgexercises.com/questions/aggregates/fachours.html


In [ ]:
df = (
    bookings
    .groupBy("facid")
    .agg(_sum("slots").alias("slots"))
    .orderBy("facid")
)
display(df)


## Aggregation 3

### Question

Produce a list of the total number of slots booked per facility in the month of September
2012. Produce an output table consisting of facility id and slots, sorted by the number of
slots.

https://pgexercises.com/questions/aggregates/fachoursbymonth.html


In [ ]:
df = (
    bookings
    .filter((col("starttime") >= "2012-09-01") & (col("starttime") < "2012-10-01"))
    .groupBy("facid")
    .agg(_sum("slots").alias("Total Slots"))
    .orderBy("Total Slots")
)
display(df)


## Aggregation 4

### Question

Produce a list of the total number of slots booked per facility per month in the year of
2012. Produce an output table consisting of facility id and slots, sorted by the id and month.

https://pgexercises.com/questions/aggregates/fachoursbymonth2.html


In [ ]:
df = (
    bookings
    .filter(year(col("starttime")) == 2012)
    .withColumn("month", month(col("starttime")))
    .groupBy("facid", "month")
    .agg(_sum("slots").alias("Total Slots"))
    .orderBy("facid", "month")
)
display(df)


## Aggregation 5

### Question

Find the total number of members who have made at least one booking.

https://pgexercises.com/questions/aggregates/members1.html


In [ ]:
df = bookings.select(countDistinct("memid").alias("count"))
display(df)


## Aggregation 6

### Question

Produce a list of each member name, id, and their first booking after September 1st 2012.
Order by member ID.

https://pgexercises.com/questions/aggregates/nbooking.html


In [ ]:
firstvisit = (
    bookings
    .filter(col("starttime") >= "2012-09-01")
    .groupBy("memid")
    .agg(_min("starttime").alias("starttime"))
)

df = (
    members
    .join(firstvisit, "memid")
    .select("surname", "firstname", "memid", "starttime")
    .orderBy("memid")
)
display(df)


## String & Date 1

### Question

Output the names of all members, formatted as 'Surname, Firstname'.

https://pgexercises.com/questions/string/concat.html


In [ ]:
df = members.select(concat_ws(", ", col("surname"), col("firstname")).alias("name"))
display(df)


## String & Date 2

### Question

Perform a case-insensitive search to find all facilities whose name begins with 'tennis'.
Retrieve all columns.

https://pgexercises.com/questions/string/case.html


In [ ]:
df = facilities.filter(lower(col("name")).startswith("tennis"))
display(df)


## String & Date 3

### Question

You've noticed that the club's member table has telephone numbers with very inconsistent
formatting. You'd like to find all the telephone numbers that contain parentheses, returning
the member ID and telephone number sorted by member ID.

https://pgexercises.com/questions/string/reg.html


In [ ]:
df = (
    members
    .filter(col("telephone").like("(%"))
    .select("memid", "telephone")
    .orderBy("memid")
)
display(df)


## String & Date 4

### Question

You'd like to produce a count of how many members you have whose surname starts with each
letter of the alphabet. Sort by the letter, and don't worry about printing out a letter if
the count is 0.

https://pgexercises.com/questions/string/substr.html


In [ ]:
df = (
    members
    .withColumn("letter", substring(col("surname"), 1, 1))
    .groupBy("letter")
    .agg(count("*").alias("count"))
    .orderBy("letter")
)
display(df)


## String & Date 5

### Question

Produce a list of all the dates in October 2012. They can be output as a timestamp (with
time set to midnight) or a date.

https://pgexercises.com/questions/date/series.html


In [ ]:
df = spark.range(1).select(
    explode(
        sequence(to_date(lit("2012-10-01")), to_date(lit("2012-10-31")), expr("interval 1 day"))
    ).alias("ts")
)
display(df)


## String & Date 6

### Question

Return a count of bookings for each month, sorted by month.

https://pgexercises.com/questions/date/bookingspermonth.html


In [ ]:
df = (
    bookings
    .withColumn("month", date_trunc("month", col("starttime")))
    .groupBy("month")
    .agg(count("*").alias("count"))
    .orderBy("month")
)
display(df)
